# Notebook 02: YOLOv8m Training on Stratified Dataset

**Retail Shelf Detection -- ShelfVision AI**

This notebook:
1. Loads the stratified dataset from Notebook 01
2. Trains YOLOv8m at 640x640 with enhanced augmentation
3. Validates on the stratified test set
4. Compares against all 11 experiments
5. Exports weights + results for deployment

**Prerequisites:**
- Run Notebook 01 first (in the same session) OR upload stratified dataset separately
- GPU accelerator enabled (T4 or better)


## Cell 1: Setup


In [ ]:
!pip install ultralytics>=8.3.0 -q

import yaml
import json
import torch
import shutil
from pathlib import Path
from ultralytics import YOLO

# ---- CONFIGURE PATHS HERE ----
# Google Colab:
WORKING_DIR = Path("/content/working")

# Kaggle:
# WORKING_DIR = Path("/kaggle/working")

OUTPUT_DIR = str(WORKING_DIR / "runs")

# Check GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

# ---- Dataset paths ----
# Option A: Notebook 01 ran in the same session
STRATIFIED_DIR = WORKING_DIR / "stratified_dataset"

# Option B: Uploaded as a separate dataset
if not STRATIFIED_DIR.exists():
    STRATIFIED_DIR = Path("/kaggle/input/stratified-retail-shelf-dataset")

# Option C: Google Drive
if not STRATIFIED_DIR.exists():
    STRATIFIED_DIR = Path("/content/drive/MyDrive/yolov11-object-detection/stratified_dataset")

assert STRATIFIED_DIR.exists(), f"Dataset not found at {STRATIFIED_DIR}"

print(f"\nDataset: {STRATIFIED_DIR}")
for split in ['train', 'valid', 'test']:
    imgs = list((STRATIFIED_DIR / split / "images").glob("*.jpg"))
    print(f"   {split}: {len(imgs)} images")

## Cell 2: Configure data.yaml


In [ ]:
# Check if data.yaml already exists from Notebook 01
DATA_YAML = WORKING_DIR / "data_stratified.yaml"

if not DATA_YAML.exists():
    # Find original data.yaml for class info
    for candidate in [
        STRATIFIED_DIR / "data.yaml",
        STRATIFIED_DIR.parent / "dataset" / "data.yaml",
    ]:
        if candidate.exists():
            with open(candidate, 'r') as f:
                cfg = yaml.safe_load(f)
            break
    else:
        raise FileNotFoundError("Cannot find data.yaml")

    cfg['train'] = str(STRATIFIED_DIR / "train" / "images")
    cfg['val'] = str(STRATIFIED_DIR / "valid" / "images")
    cfg['test'] = str(STRATIFIED_DIR / "test" / "images")

    with open(DATA_YAML, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

with open(DATA_YAML, 'r') as f:
    cfg = yaml.safe_load(f)

print(f"data.yaml configured:")
print(f"   Train: {cfg.get('train', 'N/A')}")
print(f"   Val:   {cfg.get('val', 'N/A')}")
print(f"   Test:  {cfg.get('test', 'N/A')}")
print(f"   Classes: {cfg['nc']}")

## Cell 3: Train YOLOv8m @ 640x640

This is the configuration that produced the **best results** across all 11 experiments:
- Precision: 81.15% | Recall: 92.75% | mAP@50: 96.00% | mAP@50-95: 69.57%


In [ ]:
print("=" * 65)
print("  TRAINING: YOLOv8m @ 640x640 + Enhanced Augmentation")
print("  on Stratified Dataset (80/10/10 split + oversampled)")
print("=" * 65)

model = YOLO("yolov8m.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project=OUTPUT_DIR,
    name="yolov8m_stratified",
    exist_ok=True,
    verbose=True,

    # Optimized hyperparameters
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    # Enhanced augmentation
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    degrees=15.0,
    scale=0.5,
    shear=5.0,
    flipud=0.1,
    fliplr=0.5,
    translate=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

print("\nTraining complete!")

## Cell 4: Validate on Test Set


In [ ]:
print("=" * 65)
print("  VALIDATION ON TEST SET")
print("=" * 65)

metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    project=OUTPUT_DIR,
    name="yolov8m_stratified_test",
    exist_ok=True,
)

p = metrics.box.mp
r = metrics.box.mr
m50 = metrics.box.map50
m5095 = metrics.box.map

print(f"\n{'='*60}")
print(f"  YOLOv8m STRATIFIED+OVERSAMPLE -- TEST RESULTS")
print(f"{'='*60}")
print(f"  Precision:   {p*100:.2f}%")
print(f"  Recall:      {r*100:.2f}%")
print(f"  mAP@50:      {m50*100:.2f}%")
print(f"  mAP@50-95:   {m5095*100:.2f}%")
print(f"{'='*60}")

## Cell 5: Compare Against All Previous Experiments


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# All 11 experiments
experiments = [
    {"label": "v11n",           "precision": 74.87, "recall": 66.47, "map50": 76.87, "map50_95": 46.38},
    {"label": "v11n+LowConf",   "precision": 67.84, "recall": 66.90, "map50": 69.84, "map50_95": 44.86},
    {"label": "v11n+Aug",       "precision": 76.88, "recall": 68.78, "map50": 79.21, "map50_95": 33.54},
    {"label": "v11m",           "precision": 80.99, "recall": 76.58, "map50": 84.52, "map50_95": 50.89},
    {"label": "v11m@1280",      "precision": 72.68, "recall": 85.62, "map50": None,  "map50_95": None},
    {"label": "DETR-L",         "precision": 65.55, "recall": 85.76, "map50": 88.36, "map50_95": 54.09},
    {"label": "DETR-X",         "precision": 63.67, "recall": 85.56, "map50": 86.93, "map50_95": 47.75},
    {"label": "v8m+Strat+OS",   "precision": 81.15, "recall": 92.75, "map50": 96.00, "map50_95": 69.57},
    {"label": "v11m+Strat+OS",  "precision": 81.43, "recall": 90.42, "map50": 93.48, "map50_95": 64.11},
    {"label": "v11m+SGD",       "precision": 82.48, "recall": 88.98, "map50": 93.68, "map50_95": 58.32},
    {"label": "v8m+SGD noOS",   "precision": 83.45, "recall": 89.39, "map50": 95.66, "map50_95": 67.85},
]

# Print comparison table
print(f"\n{'='*85}")
print(f"  ALL EXPERIMENTS -- COMPARISON")
print(f"{'='*85}")
print(f"  {'#':<4} {'Model':<16} {'Precision':>10} {'Recall':>8} {'mAP@50':>8} {'mAP@50-95':>10}")
print(f"  {'-'*62}")
for i, exp in enumerate(experiments, 1):
    p_str = f"{exp['precision']:.2f}%"
    r_str = f"{exp['recall']:.2f}%"
    m_str = f"{exp['map50']:.2f}%" if exp['map50'] else "--"
    m95_str = f"{exp['map50_95']:.2f}%" if exp['map50_95'] else "--"
    marker = " <-- BEST" if exp['label'] == 'v8m+Strat+OS' else ""
    print(f"  {i:<4} {exp['label']:<16} {p_str:>10} {r_str:>8} {m_str:>8} {m95_str:>10}{marker}")
print(f"{'='*85}")

# Precision vs Recall comparison chart
fig, ax = plt.subplots(figsize=(16, 7))
labels = [e['label'] for e in experiments]
prec = [e['precision'] for e in experiments]
rec = [e['recall'] for e in experiments]
x = np.arange(len(labels))

bars1 = ax.bar(x - 0.2, prec, 0.35, label='Precision', color='#3498db', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + 0.2, rec, 0.35, label='Recall', color='#2ecc71', edgecolor='white', linewidth=0.5)

# Highlight best model (Experiment #8)
bars1[7].set_color('#1a5276')
bars1[7].set_edgecolor('#3498db')
bars1[7].set_linewidth(2)
bars2[7].set_color('#196f3d')
bars2[7].set_edgecolor('#2ecc71')
bars2[7].set_linewidth(2)

ax.set_xlabel('Experiment', fontsize=12, fontweight='bold')
ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Precision vs Recall -- All 11 Experiments', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_ylim(50, 100)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(WORKING_DIR / "experiment_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved!")

## Cell 6: Export Weights & Results


In [ ]:
from IPython.display import FileLink

# Save experiment result as JSON
result_data = {
    "model": "YOLOv8m",
    "imgsz": 640,
    "dataset": "stratified + oversample",
    "split": "80/10/10 stratified",
    "augmentation": "enhanced",
    "precision": round(p * 100, 2),
    "recall": round(r * 100, 2),
    "map50": round(m50 * 100, 2),
    "map50_95": round(m5095 * 100, 2),
    "notes": "Best model -- YOLOv8m on stratified split with oversampled minorities"
}
with open(WORKING_DIR / "stratified_result.json", 'w') as f:
    json.dump(result_data, f, indent=2)

# Zip weights + results
run_dir = Path(OUTPUT_DIR) / "yolov8m_stratified"
weights_dir = run_dir / "weights"

output_zip_dir = WORKING_DIR / "export"
output_zip_dir.mkdir(exist_ok=True)

# Copy key files
if weights_dir.exists():
    shutil.copy2(weights_dir / "best.pt", output_zip_dir / "best.pt")
    shutil.copy2(weights_dir / "last.pt", output_zip_dir / "last.pt")
shutil.copy2(WORKING_DIR / "stratified_result.json", output_zip_dir / "result.json")

# Training curves
for f in run_dir.glob("*.png"):
    shutil.copy2(f, output_zip_dir / f.name)

shutil.make_archive(str(WORKING_DIR / "yolov8m_stratified_export"), 'zip', str(output_zip_dir))

print("Export complete!")
print(f"\nContains:")
for f in sorted(output_zip_dir.iterdir()):
    size = f.stat().st_size / 1024
    print(f"   {f.name} ({size:.0f} KB)")

print("\nDownload:")
display(FileLink("yolov8m_stratified_export.zip"))

print(f"\n{'='*60}")
print(f"  NEXT STEP:")
print(f"  Download best.pt and place it in your project:")
print(f"  retail-shelf-detection/models/best.pt")
print(f"  Then run: docker compose up --build")
print(f"{'='*60}")